In [0]:
create or replace temporary view v_distinct_customers as
select distinct * from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id asc;

In [0]:
create table if not exists gizmobox.silver.customers as
with cte as (select customer_id,max(created_timestamp) from v_distinct_customers
group by customer_id)
select cast(created_timestamp as timestamp) created_timestamp,
customer_id,
customer_name,
cast(date_of_birth as date) date_of_birth,
email,
cast(member_since as date) member_since,
telephone
from v_distinct_customers where (customer_id,created_timestamp) in (select * from cte)

In [0]:
select * from gizmobox.silver.customers

## Payment Data to Silver

In [0]:
REFRESH TABLE gizmobox.bronze.payments;

In [0]:
create table gizmobox.silver.payments as
select payment_id,
order_id,
cast(date_format(payment_timestamp,'yyyy-MM-dd') as date) payment_date,
date_format(payment_timestamp,'HH:mm:ss') payment_time,
case payment_status when 1 then 'Success'
    when 2 then 'Pending'
    when 3 then 'Cancelled'
    when 4 then 'Failed' end as Payment_Status,
payment_method
from gizmobox.bronze.payments;

In [0]:
select * from gizmobox.silver.payments

## Transform Refund Data to Silver

In [0]:
create table if not exists gizmobox.silver.refunds
as select 
refund_id,payment_id,
date_format(refund_timestamp,'yyyy-MM-dd') refund_date,
date_format(refund_timestamp,'HH:mm:ss') refund_time,
refund_amount,
regexp_extract(refund_reason,'^([^:]+):',1) refund_source,
split(refund_reason,':')[1] refund_reason
from azure_gizmobox_catalog_sql.dbo.refunds;

In [0]:
desc extended gizmobox.silver.refunds

In [0]:
select * from gizmobox.silver.refunds;

## Transform Membership Data to Silver

In [0]:
create table gizmobox.silver.memberships as
select regexp_extract(path,'.*/([0-9]+)\\.png$',1) customer_id,content membership_card from gizmobox.bronze.v_memberships;

In [0]:
select * from gizmobox.silver.memberships

## Transform Addresses Data to Silver

In [0]:
create table gizmobox.silver.addresses as
select * from (select customer_id,address_type,address_line_1,city,state,postcode
from gizmobox.bronze.v_addresses)
pivot (
        max(address_line_1) as address_line1,
        max(city) as city,
        max(state) as state,
        max(postcode) as postcode 
        for address_type in ('shipping','billing'));

In [0]:
select * from gizmobox.silver.addresses ;

In [0]:
select value:order_id,
        value:items.::
        ,value from gizmobox.bronze.v_orders;

In [0]:
create or replace temp view tv_orders_fixed as
select value,
       regexp_replace(value,'"order_date:" (\\{d4}-\\{d2}-\\{d2})','"order_date":"\\$1"') as fixed_value
       from gizmobox.bronze.v_orders

In [0]:
select schema_of_json(fixed_value),fixed_value as schema from tv_orders_fixed limit 1

In [0]:
drop table if exists gizmobox.silver.orders_json;
create table if not exists gizmobox.silver.orders_json as
select from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed

In [0]:
select * from gizmobox.silver.orders_json